In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import Stream



netflix2022 = pd.read_csv("../content_libraries/netflix2022.csv")
amazon2022 = pd.read_csv("../content_libraries/prime2022.csv")
netflix2023 = pd.read_csv("../content_libraries/netflix2023.csv")
amazon2023 = pd.read_csv("../content_libraries/prime2023.csv")
netflix2026 = pd.read_csv("../content_libraries/netflix2026.csv")
amazon2026 = pd.read_csv("../content_libraries/prime2026.csv")


analysis_data = pd.read_csv("../content_libraries/prices/inflation_adjusted_price.csv")
analysis_data["date"] = pd.to_datetime(analysis_data["date"])

# 1. Count titles with IMDb score >= 7
datasets = {
    ("Netflix", 2022): netflix2022,
    ("Prime Video", 2022): amazon2022,
    ("Netflix", 2023): netflix2023,
    ("Prime Video", 2023): amazon2023,
    ("Netflix", 2026): netflix2026,
    ("Prime Video", 2026): amazon2026
}

title_counts = pd.DataFrame([
    {
        "service": service,
        "year": year,
        "high_rated_titles": (df["imdb_score"] >= 7).sum()
    }
    for (service, year), df in datasets.items()
])

# 2. Select inflation-adjusted prices for the correct months
selected_months = {
    2022: 7,   # July
    2023: 3,   # March
    2026: 1    # January
}

prices = analysis_data[
    analysis_data.apply(
        lambda row: row["date"].month == selected_months[row["year"]],
        axis=1
    )
][["service", "year", "inflation_adjusted_price"]].copy()

# 3. Combine title counts with prices
value_for_money = title_counts.merge(
    prices,
    on=["service", "year"],
    how="left",
    validate="one_to_one"
)

# Check that every dataset has a matching price
if value_for_money["inflation_adjusted_price"].isna().any():
    raise ValueError("Missing price for one or more datasets.")

# 4. Calculate high-rated titles per dollar
value_for_money["titles_per_dollar"] = (
    value_for_money["high_rated_titles"]
    / value_for_money["inflation_adjusted_price"]
)

value_for_money = value_for_money.round(2)

display(value_for_money)

Netflix 2022: 2070 titles
Amazon 2022: 2202 titles
Netflix 2023: 2207 titles
Amazon 2023: 2482 titles
Netflix 2026: 2814 titles
Amazon 2026: 2369 titles
